In [3]:
"""
Full pipeline: all-year AEF features → train LightGBM → predict test → GeoJSON submission
// eas
"""

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
import lightgbm as lgb
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import sys, json

sys.path.insert(0, ".")
from submission_utils import raster_to_geojson

# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")
PGT_DIR = Path("eda_artifacts/pseudo_gt_codex")
MODEL_DIR = Path("eda_artifacts/model"); MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR = Path("submission"); SUB_DIR.mkdir(parents=True, exist_ok=True)
TILE_PRED_DIR = Path("eda_artifacts/predictions"); TILE_PRED_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]   # all 6 years
TARGET_SHAPE = (1000, 1000)
N_AEF_BANDS = 64
N_FEATURES = len(YEARS) * N_AEF_BANDS + (len(YEARS) - 1)    # 384 + 5 = 389

NEG_RATIO = 5
N_BOOST_ROUND = 400
CV_HOLDOUT = "18NXH_6_8"   # held-out validation tile


# ============================================================
# HELPERS
# ============================================================
def get_ref(tile, split):
    """Reference grid for tile (tries S2, then AEF)."""
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"
    if s2_dir.exists():
        best, best_area = None, 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                if h >= 300 and w >= 300 and h * w > best_area:
                    best_area = h * w
                    best = (r.transform, r.crs, r.shape)
        if best is not None:
            return best
    # Fallback: AEF
    aef_files = sorted((ROOT / f"aef-embeddings/{split}").glob(f"{tile}_*.tiff"))
    if aef_files:
        with rasterio.open(aef_files[0]) as r:
            return r.transform, r.crs, r.shape
    return None


def load_aef_to_ref(tile, year, split, ref):
    """Load AEF, reproject all 64 bands to reference grid."""
    p = ROOT / f"aef-embeddings/{split}/{tile}_{year}.tiff"
    if not p.exists(): return None
    
    aef = np.zeros((N_AEF_BANDS, *ref[2]), dtype=np.float32)
    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        raw = np.where(np.isfinite(raw), raw, 0.0)
        for b in range(N_AEF_BANDS):
            reproject(raw[b], aef[b],
                      src_transform=src.transform, src_crs=src.crs,
                      dst_transform=ref[0], dst_crs=ref[1],
                      resampling=Resampling.bilinear)
    return aef


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape[-2:]
    h_tgt, w_tgt = target_shape
    row_idx = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    col_idx = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)
    if arr.ndim == 2:
        return arr[row_idx[:, None], col_idx[None, :]]
    return arr[:, row_idx[:, None], col_idx[None, :]]


def extract_features(tile, split):
    """
    Returns features (N_pixels, N_FEATURES) and reference grid for tile.
    Features: 6×64 AEF values + 5 year-over-year cosine similarities.
    """
    ref = get_ref(tile, split)
    if ref is None: return None, None
    
    # Load all years
    aef_stack = []
    for year in YEARS:
        aef = load_aef_to_ref(tile, year, split, ref)
        if aef is None:
            print(f"    Missing AEF for {tile} year {year}")
            return None, None
        aef = resize_nearest(aef, TARGET_SHAPE)
        aef_stack.append(aef)
    
    # Year-over-year cosine similarities
    changes = []
    for i in range(len(YEARS) - 1):
        a, b = aef_stack[i], aef_stack[i + 1]
        dot = (a * b).sum(axis=0)
        norm_a = np.linalg.norm(a, axis=0)
        norm_b = np.linalg.norm(b, axis=0)
        denom = norm_a * norm_b
        cos_sim = np.where(denom > 1e-6, dot / (denom + 1e-9), 0.0).astype(np.float32)
        changes.append(cos_sim)
    
    # Stack all features: shape (N_FEATURES, H, W) → (N_pixels, N_FEATURES)
    all_feats = np.concatenate(
        aef_stack + [c[None] for c in changes],
        axis=0
    )
    features = all_feats.reshape(N_FEATURES, -1).T
    return features, ref


def feature_names():
    names = []
    for y in YEARS:
        for b in range(N_AEF_BANDS):
            names.append(f"aef_{y}_{b}")
    for i in range(len(YEARS) - 1):
        names.append(f"change_{YEARS[i]}_{YEARS[i+1]}")
    return names


# ============================================================
# STEP 1 — Extract features for training
# ============================================================
print(f"=== Step 1: Extract features (target = {N_FEATURES} per pixel) ===")
train_tiles = sorted([p.name.replace("__s2_l2a", "")
                      for p in (ROOT / "sentinel-2/train").iterdir()])

all_feats, all_labels, all_weights, all_tiles = [], [], [], []
for tile in tqdm(train_tiles, desc="train features"):
    pgt_path = PGT_DIR / f"{tile}.npz"
    if not pgt_path.exists(): continue
    
    feats, ref = extract_features(tile, "train")
    if feats is None: continue
    
    pgt = np.load(pgt_path)
    label = pgt["label"].flatten()
    conf = pgt["confidence"].flatten()
    
    # Keep only labeled pixels
    valid = ~np.isnan(label)
    all_feats.append(feats[valid])
    all_labels.append(label[valid])
    all_weights.append(conf[valid])
    all_tiles.append(np.full(int(valid.sum()), tile))

X = np.concatenate(all_feats, axis=0)
y = np.concatenate(all_labels).astype(np.float32)
w = np.concatenate(all_weights).astype(np.float32)
tile_ids = np.concatenate(all_tiles)

print(f"\nTotal labeled pixels: {len(y):,}")
print(f"Positives: {int((y == 1).sum()):,} ({(y == 1).mean():.2%})")
print(f"Negatives: {int((y == 0).sum()):,}")
print(f"Feature matrix: {X.shape}, dtype={X.dtype}")
print(f"Memory: {X.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 2 — Subsample negatives
# ============================================================
print(f"\n=== Step 2: Subsample negatives {NEG_RATIO}x ===")
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]
n_neg_keep = min(len(neg_idx), len(pos_idx) * NEG_RATIO)
rng = np.random.default_rng(42)
neg_sampled = rng.choice(neg_idx, size=n_neg_keep, replace=False)
keep = np.concatenate([pos_idx, neg_sampled])
rng.shuffle(keep)

X_kept = X[keep]
y_kept = y[keep]
w_kept = w[keep]
tiles_kept = tile_ids[keep]

print(f"After subsampling: {len(y_kept):,} rows ({(y_kept == 1).mean():.1%} positive)")
print(f"Memory: {X_kept.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 3 — Held-out validation for threshold
# ============================================================
val_mask = tiles_kept == CV_HOLDOUT
tr_mask = ~val_mask
X_tr = X_kept[tr_mask]; y_tr = y_kept[tr_mask]; w_tr = w_kept[tr_mask]
X_val = X_kept[val_mask]; y_val = y_kept[val_mask]

print(f"\nValidation tile: {CV_HOLDOUT}")
print(f"  Train rows: {len(y_tr):,}, Val rows: {len(y_val):,}")


# ============================================================
# STEP 4 — Train LightGBM
# ============================================================
print("\n=== Step 4: Train LightGBM ===")
names = feature_names()

lgb_train = lgb.Dataset(X_tr, y_tr, weight=w_tr, feature_name=names)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
}

model = lgb.train(
    params, lgb_train,
    num_boost_round=N_BOOST_ROUND,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

model.save_model(str(MODEL_DIR / "lgbm_allyears.txt"))


# ============================================================
# STEP 5 — Tune threshold
# ============================================================
print("\n=== Step 5: Tune threshold ===")
val_pred = model.predict(X_val)
best_thr, best_f1 = 0.15, 0

# ============================================================
# STEP 6 — Predict test tiles
# ============================================================
print("\n=== Step 6: Predict test tiles ===")
test_tiles = sorted([p.name.replace("__s2_l2a", "")
                     for p in (ROOT / "sentinel-2/test").iterdir()])
print(f"Test tiles: {test_tiles}")

all_features_geojson = []

for tile in tqdm(test_tiles, desc="predicting"):
    feats, ref = extract_features(tile, "test")
    if feats is None:
        print(f"  {tile}: skipped")
        continue
    
    pred = model.predict(feats)
    binary = (pred > best_thr).astype(np.uint8).reshape(TARGET_SHAPE)
    
    # Resize binary to native resolution for georeferencing
    h_native, w_native = ref[2]
    h_tgt, w_tgt = TARGET_SHAPE
    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_native).clip(0, w_tgt - 1)
    binary_native = binary[row_idx[:, None], col_idx[None, :]]
    
    # Save GeoTIFF
    tile_path = TILE_PRED_DIR / f"{tile}_pred.tif"
    profile = {
        "driver": "GTiff", "height": h_native, "width": w_native,
        "count": 1, "dtype": "uint8",
        "crs": ref[1], "transform": ref[0], "nodata": 0,
    }
    with rasterio.open(tile_path, "w", **profile) as dst:
        dst.write(binary_native, 1)
    
    # Convert to GeoJSON (drops polygons < 0.5 ha inside submission_utils)
    geojson = raster_to_geojson(str(tile_path), output_path=None)
    for feat in geojson["features"]:
        feat["properties"]["tile"] = tile
    all_features_geojson.extend(geojson["features"])
    print(f"  {tile}: {int(binary_native.sum()):,} positive pixels, "
          f"{len(geojson['features'])} polygons")


# ============================================================
# STEP 7 — Save submission
# ============================================================
submission = {"type": "FeatureCollection", "features": all_features_geojson}

sub_path = SUB_DIR / "submission_first.geojson"
with open(sub_path, "w") as f:
    json.dump(submission, f)

print(f"\n=== Submission saved: {sub_path} ===")
print(f"Total polygons: {len(all_features_geojson)}")


# ============================================================
# STEP 8 — Feature importance (top 15)
# ============================================================
print("\n=== Feature importance (top 15) ===")
importance = pd.DataFrame({
    "feature": names,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)
print(importance.head(15).to_string(index=False))

# Also aggregate by year to see which year's AEF matters most
print("\n=== Importance by year ===")
by_year = {y: 0 for y in YEARS}
change_total = 0
for _, row in importance.iterrows():
    feat = row["feature"]
    if feat.startswith("aef_"):
        year = int(feat.split("_")[1])
        by_year[year] += row["gain"]
    elif feat.startswith("change_"):
        change_total += row["gain"]

for y in YEARS:
    print(f"  AEF {y}: {by_year[y]:.0f}")
print(f"  All changes: {change_total:.0f}")

Retraining on ALL sampled tiles.


NameError: name 'y_kept' is not defined